# pytauargus — Quick Start

**Statistical Disclosure Control (SDC) for tabular data**, as a
Python package and a headless `tauargus` command. This notebook
starts with a DataFrame: `protect()` → `TableResult` → masked output.
It then shows how existing metadata and `.arb` batches fit the same engine.

## How to run

```bash
# from the bindings/python/ directory
uv sync --extra notebooks
uv run --with jupyterlab jupyter lab notebooks/01_quickstart.ipynb
```

Or headless (no Jupyter needed):

```bash
uv run --with nbclient --with nbformat --with ipykernel \
  python notebooks/_run.py 01_quickstart.ipynb
```

## What you need

- `pytauargus>=0.3.0` for the DataFrame `protect()` API; published
  wheels include the native engine and HiGHS. For source checkouts,
  follow the [build instructions](../../../README.md#building-from-source).
- The repository's `data/` fixtures (included in the repo)
- Python 3.10–3.13; macOS arm64 / Linux x86_64+aarch64 / Windows x64
- **No licensed solver, no SPSS, no external executable** — the LP/MIP
  solver (HiGHS) is bundled in the wheel

---

In [1]:
from pathlib import Path
import pytauargus

# Locate the repo's data/ fixtures (works from notebooks/ or repo root)
_here = Path.cwd()
DATA = next(
    (p / "data") for p in [_here, *_here.parents]
    if (p / "data" / "tau_testW.asc").exists()
)
OUT = _here / "out"          # scratch dir for notebook outputs (gitignored)
OUT.mkdir(exist_ok=True)

def display_paths(text):
    """Use portable paths in displayed text; engine inputs keep real paths."""
    text = str(text)
    for path, label in ((OUT, "./out"), (DATA, "<repo>/data")):
        text = text.replace(str(path), label).replace(path.as_posix(), label)
    return text

print(f"pytauargus {pytauargus.__version__}")
print("data : <repo>/data")
print("out  : ./out")

pytauargus 0.2.1
data : <repo>/data
out  : ./out


## 1. Protect a DataFrame

Each row below is one respondent. `NK(2,75)` marks a cell as unsafe
when its two largest contributors exceed 75% of the response.
`protect()` writes `.asc`, `.rda` and `.arb`, computes the crossing,
applies OPT suppression by default and reads the result back.

In [2]:
import pandas as pd
from pytauargus import protect

# Twelve synthetic respondents; North/Retail has one dominant contributor.
micro = pd.DataFrame({
    "Region": ["North"] * 6 + ["South"] * 6,
    "Sector": ["Retail"] * 3 + ["Services"] * 3
              + ["Retail"] * 3 + ["Services"] * 3,
    "Turnover": [100., 5., 5., 10., 10., 10., 20., 20., 20., 30., 30., 30.],
})
micro

,Region,Sector,Turnover
0,North,Retail,100.0
1,North,Retail,5.0
2,North,Retail,5.0
3,North,Services,10.0
4,North,Services,10.0
5,North,Services,10.0
6,South,Retail,20.0
7,South,Retail,20.0
8,South,Retail,20.0
9,South,Services,30.0


In [3]:
protected = protect(
    micro, ["Region", "Sector"], response="Turnover",
    safety_rules="NK(2,75)", workdir=str(OUT / "01_protect"),
)
table = protected.tables[0]
assert table.n_cells == 9  # 2 x 2 crossing, including row/column totals
print(table)
print("status:", table.status())

TableResult(n_cells=9, response='Turnover', exp=['Region', 'Sector'], is_freq=False)
status: ['S', 'S', 'S', 'U', 'U', 'S', 'M', 'M', 'S']


## 2. Build the masked publication table

`TableResult.dataframe()` is the full diagnostic result and includes
original responses. `unsafe()` returns those unmasked values.
For publication, select the explanatory codes and use `safe()` for
the response: primary (`U`) and secondary (`M`) cells become `x`.
Use `safe(None)` if your downstream format expects missing values.

In [4]:
published = table.dataframe()[["Region", "Sector"]].copy()
published["Turnover"] = table.safe()
assert all(value == "x" for status, value in zip(table.status(), table.safe())
           if status in {"U", "M"})
published

,Region,Sector,Turnover
0,Total,Total,290.0
1,Total,Retail,170.0
2,Total,Services,120.0
3,North,Total,x
4,North,Retail,x
5,North,Services,30.0
6,South,Total,x
7,South,Retail,x
8,South,Services,90.0


### Results and intermediate files

`ProtectResult.tables` holds one result per table; `.engine` gives
access to audit and other engine operations. `.files` and `.workdir`
locate the generated inputs and type-5 `.tab` results.
The files and diagnostic values are analyst artifacts. Keep them
separate from the masked publication table.

In [5]:
print("files:", {key: Path(path).name for key, path in protected.files.items()})
print("original response:", table.unsafe())  # synthetic analyst data only
published.to_csv(OUT / "01_dataframe_publication.csv", index=False)
print("wrote 01_dataframe_publication.csv")

files: {'asc': 'micro.asc', 'rda': 'micro.rda', 'arb': 'batch.arb', 'tab1': 'tab1.tab'}
original response: [290.0, 170.0, 120.0, 140.0, 110.0, 30.0, 150.0, 60.0, 90.0]
wrote 01_dataframe_publication.csv


## Working with existing batches

pytauargus has **zero runtime dependencies**; the native engine
(C/C++ port of the Tau-Argus core, built against HiGHS) ships inside
the wheel. The main entry points:

| Symbol | Source | Purpose |
|--------|--------|---------|
| `protect(…)` / `ProtectResult` | `pytauargus` | DataFrame/dict → protected tables + files + engine |
| `TableResult` | `pytauargus` | `safe()`, `status()`, `unsafe()`, `dataframe()` |
| `Engine` | `pytauargus.engine` | High-level driver for the native engine |
| `run_batch(arb)` | `pytauargus.engine` | Parse + execute a `.arb` batch file → `Engine` |
| `parse_rda(path)` | `pytauargus.engine` | Parse `.rda` metadata → `Metadata` |
| `parse_batch(path)` | `pytauargus.batch` | Parse `.arb` → list of `Command` dataclasses |
| `micro_arb(…)` | `pytauargus.arb` | **Generate** a `.arb` batch file |
| `write_rda` / `rda_text` | `pytauargus.rda` | **Generate** `.rda` metadata text |
| `write_hrc(…)` | `pytauargus.hrc` | **Generate** a `.hrc` hierarchy file |
| `Suppress`, `WriteTable`, `SpecifyTable`, … | `pytauargus.batch` | Parsed command dataclasses |

### Repository fixtures

The repo ships the legacy Tau-Argus 4.1 test dataset (the same files
used by the legacy product's documentation):

- `tau_testW.asc` — fixed-width microdata (one record per line)
- `tau_testW.rda` — metadata: positions, widths, types, codelists
- `TestRecode.arb` — a ready-made batch file (2 tables + recodes)

### The microdata file (first 4 lines)

Each record is a fixed-width ASCII line. Column positions and widths
are described in the `.rda` file — the engine parses each record
without any schema inference.

In [6]:
with open(DATA / "tau_testW.asc") as f:
    for i, line in enumerate(f):
        if i >= 4: break
        print(f"{i+1:3d}  {line.rstrip()}")

  1  84 34100 8  9  2.2     9272     28.26    166760    279434   2009571    367159    527645    216851 0
  2  84 29240 7 11  1.7     2901       887     61298    104155    337247     60892     49734     63413 0
  3  84 29240 7 10  3.3      302        96      6335     10750     34800      6289      5137      6551 0
  4  84 29240 5  9  2.2     1038       320     18557     31134     98448     16897     14755     18539 0


### The metadata file (first 14 lines)

The `.rda` describes each variable's position, width, missing code,
type, codelist and hierarchy. Note `Var2` — the numeric response
variable with 2 decimals that our batch tabulates.

In [7]:
with open(DATA / "tau_testW.rda") as f:
    for i, line in enumerate(f):
        if i >= 14: break
        print(f"{i+1:3d}  {line.rstrip()}")

  1  Year 1  2 99
  2    <RECODEABLE>
  3    <DISTANCE>  1  2  3  4  5
  4  IndustryCode 4  5 99999
  5    <RECODEABLE>
  6    <HIERARCHICAL>
  7    <HIERLEVELS>  3 1 1 0 0
  8    <DISTANCE>  1  3  5  7  9
  9  Size 9  2 99
 10    <RECODEABLE>
 11  Region 12  2 99
 12    <RECODEABLE>
 13    <CODELIST> "region.cdl"
 14    <HIERARCHICAL>


### Parse the metadata

`parse_rda` reads the `.rda` file and returns a `Metadata` object —
a list of `Variable` records plus file-level options.

In [8]:
from pytauargus.engine import parse_rda

meta = parse_rda(DATA / "tau_testW.rda")
print(f"{len(meta.variables)} variables")
print()
for i, v in enumerate(meta.variables):
    print(f"  {v.name:<16} type={v.type:<14}  pos={v.b_pos:>2}  width={v.var_len:>2}  dec={v.n_decimals}")

14 variables

  Year             type=CATEGORICAL     pos= 1  width= 2  dec=0
  IndustryCode     type=CATEGORICAL     pos= 4  width= 5  dec=0
  Size             type=CATEGORICAL     pos= 9  width= 2  dec=0
  Region           type=CATEGORICAL     pos=12  width= 2  dec=0
  Wgt              type=WEIGHT          pos=15  width= 4  dec=1
  Var1             type=RESPONSE        pos=19  width= 9  dec=0
  Var2             type=RESPONSE        pos=28  width=10  dec=2
  Var3             type=RESPONSE        pos=38  width=10  dec=0
  Var4             type=RESPONSE        pos=48  width=10  dec=0
  Var5             type=RESPONSE        pos=58  width=10  dec=0
  Var6             type=RESPONSE        pos=68  width=10  dec=0
  Var7             type=RESPONSE        pos=78  width=10  dec=0
  Var8             type=RESPONSE        pos=88  width=10  dec=0
  Request          type=REQUEST         pos=99  width= 1  dec=0


### `Variable` attributes (the ones you'll use)

| Attribute | Meaning |
|-----------|---------|
| `name` | Variable name (lookup is case-insensitive via `Metadata.find`) |
| `type` | `CATEGORICAL`, `RECODEABLE`, `NUMERIC`, `WEIGHT`, `HOLDING`, `REQUEST`, … |
| `b_pos` | 1-based byte position in the `.asc` record |
| `var_len` | Field width in bytes |
| `n_decimals` | Decimal places (NUMERIC / WEIGHT) |
| `missing` | Missing-value codes (up to two, padded strings) |
| `tot_code` | Total code, e.g. `"99"` |
| `code_list_file` | Path to a `.cdl` codelist file |
| `hierarchical` | `0` = none, `1` = levels, `2` = `.hrc` file |

Helper predicates: `is_categorical()`, `is_numeric()`, `is_weight()`,
`is_holding()`, `is_total_code(code)`, `is_missing(code)`.

### Run a batch

`run_batch` is the main entry point. It parses the `.arb` file,
executes every command in order, and returns the configured
`Engine` with tables computed and ready for inspection.

In [9]:
from pytauargus.engine import run_batch

eng = run_batch(DATA / "TestRecode.arb")
print(f"engine version: {eng.version}")
print(f"tables: {eng._n_tables}")
for i in range(eng._n_tables):
    ncell, _ = eng._tau.get_total_table_size(i)
    spec, _ = eng._tables[i]
    print(f"  table {i+1}: {' x '.join(spec.exp_vars):<24} | {spec.resp_var}  →  {ncell} cells")

engine version: 1.1.4.11
tables: 2
  table 1: Size x Region            | Var2  →  126 cells
  table 2: Region x IndustryCode    | Var2  →  12726 cells


### What `TestRecode.arb` contains

Two tables with NK safety rules, then hierarchical recodes — this is
the same file the legacy documentation ships:

In [10]:
print((DATA / "TestRecode.arb").read_text())

//This batch file was generated by Tau-ARGUS
//Date Thu Apr 05 22:05:17 CEST 2018
//
<OPENMICRODATA>    "tau_testW.asc"
<OPENMETADATA>     "tau_testW.rda"
<SPECIFYTABLE>     "Size""Region"|"Var2"||
<SAFETYRULE>       NK(2,75)|NK(0,0)
<SPECIFYTABLE>     "Region""IndustryCode"|"Var2"||
<SAFETYRULE>       NK(2,75)|NK(0,0)
<READMICRODATA>
<RECODE>           1, Size, "GK.grc"
<RECODE>           1, Region, 2
<RECODE>           2, IndustryCode, "IndCode2.grc"
<GOINTERACTIVE>



### The batch grammar, parsed

`parse_batch` returns one `Command` dataclass per line —
`OpenMicrodata`, `OpenMetadata`, `SpecifyTable`, `SafetyRuleSet`,
`ReadMicrodata`, `Recode`, … This is the exact grammar the engine
executes, and the dataclasses are what the generators (notebook 3)
and the Python API accept.

In [11]:
from pytauargus.batch import parse_batch

cmds = parse_batch(DATA / "TestRecode.arb")
for c in cmds:
    print(f"  {type(c).__name__:<16} {str(c)[:78]}")

  OpenMicrodata    OpenMicrodata(file='tau_testW.asc')
  OpenMetadata     OpenMetadata(file='tau_testW.rda')
  SpecifyTable     SpecifyTable(exp_vars=['Size', 'Region'], resp_var='Var2', shadow_var='', cost
  SafetyRule       SafetyRule(kind='NK', args=['2', '75'])
  SafetyRule       SafetyRule(kind='NK', args=['0', '0'])
  SpecifyTable     SpecifyTable(exp_vars=['Region', 'IndustryCode'], resp_var='Var2', shadow_var=
  SafetyRule       SafetyRule(kind='NK', args=['2', '75'])
  SafetyRule       SafetyRule(kind='NK', args=['0', '0'])
  ReadMicrodata    ReadMicrodata()
  Recode           Recode(tab_no=1, var_name='Size', recode_file='GK.grc')
  Recode           Recode(tab_no=1, var_name='Region', recode_file='2')
  Recode           Recode(tab_no=2, var_name='IndustryCode', recode_file='IndCode2.grc')
  GoInteractive    GoInteractive()


### Inspect native cells

After `run_batch`, every cell in every table has a **status** and a
**value**. Status codes (from the native engine):

| Code | Status | Meaning |
|------|--------|---------|
| 1 | `safe` | Publishable as-is |
| 2 | `safe_manual` | Manually marked safe |
| 3–9 | `unsafe_*` | Failed a safety rule (rule / peep / freq / zero / singleton) |
| 10 | `protect_manual` | Manually marked for protection |
| 11 | `secondary_unsafe` | Suppressed secondarily to protect others |
| 13 | `empty_nonstructural` | |
| 14 | `empty` | No contributing records |

In [12]:
from collections import Counter

STATUS_NAMES = {
    1: "safe", 2: "safe_manual", 3: "unsafe_rule", 4: "unsafe_peep",
    5: "unsafe_freq", 6: "unsafe_zero", 7: "unsafe_singleton",
    8: "unsafe_singleton_manual", 9: "unsafe_manual", 10: "protect_manual",
    11: "secondary_unsafe", 12: "secondary_unsafe_manual",
    13: "empty_nonstructural", 14: "empty",
}

tau = eng._tau
n = tau.get_total_table_size(0)[0]
statuses = Counter(tau.get_table_cell_status(0, j) for j in range(n))
print(f"Table 1 — {n} cells:")
for code_, count in sorted(statuses.items()):
    print(f"  {STATUS_NAMES.get(code_, code_):<24} {count}")

Table 1 — 126 cells:
  safe                     104
  unsafe_rule              1
  empty                    21


### A few individual cells

`get_table_cell(tab, dim, top_n)` returns a tuple: the response
value at index 1, the status at index 10, and (after an audit) the
realized lower/upper bounds as the last two entries. Walk the first
row-major cells of table 1:

In [13]:
spec, _ = eng._tables[0]
exp = [eng.metadata.index_of(nm) for nm in spec.exp_vars]
max_dim = [tau.get_var_number_of_codes(vi)[1] for vi in exp]   # active codes
print(f"table 1 dims: {max_dim}")
print()
dim = [0, 0]
for idx in range(6):
    res = tau.get_table_cell(0, dim, 0)
    val, status = res[1], res[10]
    print(f"  cell {idx:2d}  dims={dim}  value={val:>12.2f}  "
          f"status={STATUS_NAMES.get(status, status)}")
    dim[1] += 1
    if dim[1] >= max_dim[1]:
        dim[0] += 1
        dim[1] = 0

table 1 dims: [7, 18]

  cell  0  dims=[0, 0]  value= 16847646.84  status=safe
  cell  1  dims=[0, 1]  value=  4373664.00  status=safe
  cell  2  dims=[0, 2]  value=  1986129.00  status=safe
  cell  3  dims=[0, 3]  value=  1809246.00  status=safe
  cell  4  dims=[0, 4]  value=   578289.00  status=safe
  cell  5  dims=[0, 5]  value=  3703896.00  status=safe


### Try it yourself

- Change `DATA / "TestRecode.arb"` to another `.arb` in `data/`.
- Swap table index `0` for `1` and count its statuses.
- `tau.get_var_code(vi, ci)` → `(ok, code_type, code, missing, level)`
  decodes any categorical code.

## Summary

You've now:

1. Protected a DataFrame with `protect()` and inspected `TableResult`
2. Exported masked responses with `safe()`
3. Parsed the `.rda` metadata and listed the variables
4. Run a `.arb` batch through the engine
5. Inspected computed table cells and their safety statuses

**Next:** [02_protection_and_audit.ipynb](02_protection_and_audit.ipynb)
— suppress the unsafe cells and run the audit (Intervalle).